# Choose which components to emit: the selector (CPU)

Import this notebook into Kaggle and press **Save & Run All**. None of the cells asks for input. With the default `MODEL = "r34"` no cell needs editing. To use another model, change only `MODEL` in the settings cell and attach its cache:

| `MODEL` | Encoder | Save the training notebook as | Checkpoint |
| --- | --- | --- | --- |
| `r34` (default) | `resnet34` | `train-unet-r34-crops-80ep` | `unet_r34_crops.pt` |
| `r50` | `resnet50` | `train-unet-r50-crops` | `unet_r50_crops.pt` |
| `effb3` | `efficientnet-b3` | `train-unet-effb3-crops` | `unet_effb3_crops.pt` |
| `r34_v2` | `resnet34` | `train-unet-r34-v2` | `unet_r34_v2.pt` |
| `r34_v2_agree` | `resnet34` | `train-unet-r34-v2-agree` | `unet_r34_v2_agree.pt` |

This notebook is step B2 of `docs/ROADMAP_V2.md`. Today every component above `min_area` is emitted. Here the candidates come from a permissive segmentation, a small model predicts each candidate's expected matched IoU per annotator view from its shape, probability, position and contrast, and a candidate is emitted when the prediction is above half of the PQ that the emitted set reaches (the derivation is in section 4 of the roadmap).

It tries two candidate sets (`filament.selector.CANDIDATE_POINTS`) and two models: a shallow tree as the baseline, and a small gradient-boosted model. Nothing is tuned.

**How to read it.** All scores are out of fold: a file is scored by a model that did not see its date, under an emission threshold computed without that fold. Against the current setting, the rule is the roadmap's: a gain of at least 0.005 in at least 90% of the bootstrap resamples, in at least 4 of 5 partitions of the dates. Read each `verdict` row. If none says `ADOPT`, keep the current setting or the object filter from `sweep_object_filter.ipynb`, which is simpler. Another team's selector of this kind gained, and yet another team's lost in 4 of 5 seeds, so losing here would not be a surprise.

Comparing 2 candidate sets and 2 models is itself a small selection, so a verdict that only just passes in one of the four is weak evidence.

It reads only the holdout side of the cache. It does not train a network, does not read the test images, does not write `submission.csv` and does not submit to Kaggle. It does not save a selector: that comes only if one is adopted, for exactly this checkpoint, flips and post-processing.

**Attach exactly these Kaggle inputs**

1. The competition `filament-segmentation-2026` (train images and train JSON).
2. The dataset `ignacioaguirrecandia/solar-code` (the repo zip), a version that contains `filament.selector`. An older version stops at the imports.
3. The private dataset `solar-probs-<MODEL>`, made from the output of `export_probs.ipynb` for this model.

**Settings**

- Accelerator: **None**. This notebook does not use a GPU.
- Internet: **Off** is enough. It uses the scikit-learn that Kaggle already has, and prints its version. The repo pins 1.9.1; a different version on Kaggle can give slightly different numbers.

**Output:** `/kaggle/working/holdout_selector_<MODEL>.csv`, small and with no image or mask data, so it can go in `results/`. The running time is not measured yet [VERIFY]; a progress line is printed every 20 frames.

## 1. `src` on the path

`src` is the directory that contains `filament/crop_training.py`. A checkout in the working directory is used as it is. Otherwise the walk goes through `/kaggle/input` at any depth, skips every `competitions` directory, and takes the shallowest match. It is used in place and not copied.

In [ ]:
import os
import sys
from pathlib import Path


def _local_src() -> Path | None:
    """A checkout that already has the crop modules (a run off Kaggle)."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        src = candidate / "src"
        if (src / "filament" / "crop_training.py").is_file():
            return src
    return None


def _private_dataset_src(root: Path | None = None) -> Path | None:
    """Return the shallowest private-dataset ``src`` under ``root``.

    ``filament/crop_training.py`` may sit at any depth. Directories named
    ``competitions`` are not walked, so the MAGFiLO tree is left alone.
    """
    root = Path("/kaggle/input") if root is None else root
    if not root.is_dir():
        return None
    marker = Path("filament") / "crop_training.py"
    matches: list[Path] = []
    for dirpath, dirnames, _filenames in os.walk(root, topdown=True):
        dirnames[:] = [name for name in dirnames if name != "competitions"]
        current = Path(dirpath)
        if current.name != "src" or not (current / marker).is_file():
            continue
        matches.append(current)
        dirnames.clear()
    if not matches:
        return None
    return min(
        matches,
        key=lambda path: (len(path.relative_to(root).parts), path.as_posix()),
    )


def _put_on_path(src: Path) -> None:
    text = str(src)
    if text not in sys.path:
        sys.path.insert(0, text)
    current = os.environ.get("PYTHONPATH", "")
    parts = [part for part in current.split(os.pathsep) if part]
    if text not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([text, *parts]) if parts else text


def _prepare_src() -> Path:
    """Put the repo's ``src`` on ``sys.path`` and return it.

    A checkout in the working directory is used as it is. Otherwise the
    private dataset's ``src`` is used in place. It is not copied to
    ``/kaggle/working``: a copy there would be saved with this notebook's
    output and found again by a later notebook that attaches that output.
    """
    local = _local_src()
    if local is not None:
        _put_on_path(local)
        return local
    source = _private_dataset_src()
    if source is None:
        raise FileNotFoundError(
            "Could not find src/filament/crop_training.py in the working "
            "directory or under /kaggle/input (any depth outside "
            "competitions). Attach the dataset ignacioaguirrecandia/solar-code, "
            "for example "
            "/kaggle/input/datasets/ignacioaguirrecandia/solar-code/"
            "Solar-Filament-Segmentation-main/src. If it is attached, its "
            "version predates the crop modules: upload a new version of the "
            "dataset from the branch that has them. Do not make the GitHub "
            "repository public."
        )
    _put_on_path(source)
    return source


SRC = _prepare_src()
print(f"src={SRC}")
PYTHON = sys.executable

## 2. Settings and the cache

- `BASE` is the current setting (`docs/PROJECT_REPORT.md`, row 12). Every difference is measured against it.
- `CANDIDATE_POINTS` and the models come from `filament.selector`. They are not tuned.
- The cache has to come from exactly this checkpoint: its manifest goes through the same guard as a weights file. Leave `ALLOW_UNVERIFIED_WEIGHTS` as `False`.

In [ ]:
import csv

import sklearn

from filament.annotations import (
    TRAIN_ANNOTATIONS_PATH,
    TRAIN_IMAGES_DIR,
    load_train_annotations,
)
from filament.checkpoint import crop_checkpoint, verify_and_report
from filament.crop_data import read_gray
from filament.gap_analysis import CURRENT_POINT, holdout_dates
from filament.probcache import (
    ProbCache,
    find_cache_root,
    manifest_fingerprint,
    read_manifest,
    split_dir,
)
from filament.selector import (
    CANDIDATE_POINTS,
    SELECTOR_COLUMNS,
    SELECTOR_MODELS,
    build_file_candidates,
    evaluate_selector,
)
from filament.stats import weighted_pq
from filament.sweep import count_views, per_file_counts
from filament.train_data import split_training_images

print(f"scikit-learn {sklearn.__version__}")

# ---- Which model: a name of filament.checkpoint.CROP_CHECKPOINTS --------------
MODEL = "r34"
EXPECTED = crop_checkpoint(MODEL)

# ---- The comparison ----------------------------------------------------------
BASE = CURRENT_POINT
ALLOW_UNVERIFIED_WEIGHTS = False

INPUT_ROOT = Path("/kaggle/input")
OUTPUT_PATH = Path(f"/kaggle/working/holdout_selector_{MODEL}.csv")

CACHE_ROOT = find_cache_root(MODEL, INPUT_ROOT)
print(f"cache={CACHE_ROOT}")
verify_and_report(
    manifest_fingerprint(read_manifest(CACHE_ROOT, MODEL)),
    allow_unverified=ALLOW_UNVERIFIED_WEIGHTS,
    expected=EXPECTED,
)

## 3. Date holdout

The same split as the training notebook: the most recent 20% of calendar dates (142 files with 231 annotator views on the current data). Every holdout file has to be in the cache.

In [ ]:
IMAGE_DIR = Path(TRAIN_IMAGES_DIR)
ANNOTATIONS_PATH = Path(TRAIN_ANNOTATIONS_PATH)
if not ANNOTATIONS_PATH.is_file() or not IMAGE_DIR.is_dir():
    raise FileNotFoundError(
        "Train data was not found. Checked "
        f"images={IMAGE_DIR} and annotations={ANNOTATIONS_PATH}. "
        "Attach the competition filament-segmentation-2026 to this notebook."
    )
loaded = load_train_annotations(ANNOTATIONS_PATH)
holdout = split_training_images(loaded.images).holdout
holdout_views = count_views(holdout)
print(f"holdout files={len(holdout)} annotator views={holdout_views}")
if (len(holdout), holdout_views) != (142, 231):
    print(
        "WARNING: the holdout is not 142 files with 231 views, the numbers the "
        "training notebook used. [VERIFY] the data version."
    )
cache = ProbCache(split_dir(CACHE_ROOT, MODEL, "holdout"))
missing = [image.file_name for image in holdout if image.file_name not in cache]
if missing:
    raise RuntimeError(
        f"{len(missing)} holdout files are not in the cache, for example {missing[0]}."
    )

## 4. The base, and the candidates

The base is the per-file counts of the current setting, from the same scoring as every other notebook. Each candidate set then gets one table per file: the features of each component and which ground-truth masks it matches. The scoring of a subset of components gives exactly the counts `filament.metrics.fast_view_counts` gives on the pixels, which the tests pin.

In [ ]:
def _read(image):
    return read_gray(IMAGE_DIR / image.file_name, (image.height, image.width))


def _load(image):
    return cache.mean(image.file_name)


def _progress(index, total, image):
    if index == 1 or index % 20 == 0 or index == total:
        print(f"  {index}/{total} {image.file_name}", flush=True)


print("base")
base_counts = per_file_counts(holdout, _read, _load, [BASE], progress=_progress).counts[
    BASE
]
print(f"pooled PQ at {BASE}: {weighted_pq(base_counts):.4f}")

dates = holdout_dates(holdout)
CANDIDATES = {}
for point in CANDIDATE_POINTS:
    print(f"candidates at {point}")
    files = []
    for index, (image, day) in enumerate(zip(holdout, dates, strict=True), start=1):
        files.append(
            build_file_candidates(
                image,
                day,
                _read(image),
                cache.mean(image.file_name),
                cache.std(image.file_name),
                point,
            )
        )
        _progress(index, len(holdout), image)
    CANDIDATES[point] = files
    print(f"  {sum(item.size for item in files)} candidates in {len(files)} files")

## 5. The verdicts

One row per partition of the dates and one `verdict` row, for each candidate set and model. `diff_vs_base` is against the current setting, with a 95% interval, over the same resampled files. `emitted` says how many candidates passed.

Add one row to `docs/RESULTS.md` from the verdict rows and put the CSV in `results/`.

In [ ]:
rows = []
for point, files in CANDIDATES.items():
    for name, make_model in SELECTOR_MODELS.items():
        print(f"{point} {name}", flush=True)
        model_rows, adopted = evaluate_selector(
            files,
            base_counts,
            make_model,
            model_name=name,
            candidates_name=(
                f"threshold {point.threshold:g}, close_size {point.close_size}, "
                f"min_area {point.min_area}"
            ),
        )
        rows += model_rows
        for row in model_rows:
            print(
                f"  {row['partition']:<8} pq={row['pooled_pq']} "
                f"diff={row['diff_vs_base']} [{row['diff_low']}, {row['diff_high']}] "
                f"better={row['fraction_better']} emitted={row['emitted']} {row['note']}"
            )

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with OUTPUT_PATH.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=SELECTOR_COLUMNS)
    writer.writeheader()
    writer.writerows(rows)
print(f"wrote {OUTPUT_PATH}")
print("This notebook read no test image, wrote no submission and did not submit.")